#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [1]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive
Mounted at /content/drive


#`Setup And Update ComfyUI`



In [4]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

# 确保回到 /content 目录再进行操作
%cd /content

if DRIVE_PATH:
    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd {WORKSPACE}

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}


/content
-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 62139, done.
remote: Counting objects: 100% (387/387), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 62139 (delta 332), reused 250 (delta 249), pack-reused 61752 (from 3)
Receiving objects: 100% (62139/62139), 97.50 MiB | 20.02 MiB/s, done.
Resolving deltas: 100% (43020/43020), done.
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 160.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 139.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 133.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 125.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# `Models Download`

## Download MODELS

In [6]:
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2


from google.colab import userdata

CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
print("Loaded API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")

# Install custom nodes

def install_custom_node(url):
  %cd /content/ComfyUI/custom_nodes
  !git clone {url}

def downloadModel(url, filename = None):
  if 'huggingface.co' in url:
    if filename is None:
      filename = url.split('/')[-1]
      filename = filename.removesuffix('?download=true')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}  -o {filename}
  else:
    # civitai
    if filename:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN} -o {filename}
    else:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN}


# install custom nodes
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Impact-Pack')


# download models
%cd ./models/checkpoints

# CyberRealistic Pony
# downloadModel('https://civitai.com/api/download/models/2908999') # Disabled due to error

# User requested model
downloadModel('https://civitai.red/api/download/models/3112728?fileId=3015314')

# Additional User requested models
downloadModel('https://civitai.red/api/download/models/3263843?fileId=3147349')
downloadModel('https://civitai.red/api/download/models/2615702?fileId=2503211')
downloadModel('https://civitai.red/api/download/models/3435333?fileId=3334536')

# Protovision XL
#downloadModel('https://civitai.com/api/download/models/201514')
# Juggernaut XL Hyper
#downloadModel('https://civitai.com/api/download/models/471120')
# Juppernaut X
#downloadModel('https://civitai.com/api/download/models/456194')
# Juggernaut v8
#downloadModel('https://civitai.com/api/download/models/288982')

# Huggingface example
#downloadModel('https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors?download=true')

%cd {WORKSPACE}


/content/ComfyUI
Loaded API key: ✅
/content/ComfyUI/models/checkpoints

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
673dd5|OK  |       0B/s|/content/ComfyUI/models/checkpoints/lustifyNSFWCheckpoint_v10Krea2_3015314.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e3a3f6|OK  |   354MiB/s|/content/ComfyUI/models/checkpoints/anima_turboV11.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
5fe56c|OK  |   347MiB/s|/content/ComfyUI/models/checkpoints/hassakuXLIllustrious_v34.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================

## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [7]:
!ls -al ./models/checkpoints/

total 49696872
drwxr-xr-x  2 root root        4096 Oct 30 17:06 .
drwxr-xr-x 28 root root        4096 Oct 30 16:11 ..
-rw-r--r--  1 root root  4182230656 Oct 30 17:05 anima_turboV11.safetensors
-rw-r--r--  1 root root  6938065408 Oct 30 17:05 hassakuXLIllustrious_v34.safetensors
-rw-r--r--  1 root root 13485909944 Oct 30 17:06 krea2Anime_v30_fp8.safetensors
-rw-r--r--  1 root root 26283344816 Oct 30 16:47 lustifyNSFWCheckpoint_v10Krea2_3015314.safetensors
-rw-r--r--  1 root root           0 Oct 30 16:11 put_checkpoints_here


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [18]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-10-30 17:37:35--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64.deb [following]
--2026-10-30 17:37:35--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ed15b987-8efb-4e5e-ac4a-80851c2c72e1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-30T18%3A32%3A46Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

In [19]:
!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "torch-npu is not installed: No module named 'torch_npu'", 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'c

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [10]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-10-30 17:11:15--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64.deb [following]
--2026-10-30 17:11:16--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ed15b987-8efb-4e5e-ac4a-80851c2c72e1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-30T17%3A51%3A24Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

## CF Tunnel

In [20]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
Traceback (most recent call last):
  File "/usr/lib/python3.13/importlib/metadata/__init__.py", line 915, in read_text
    return self._path.joinpath(filename).read_text(encoding='utf-8')
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 546, in read_text
    return PathBase.read_text(self, encoding, errors, newline)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_abc.py", line 632, in read_text
    with self.open(mode='r', encoding=encoding, errors=errors, newline=newline) as f:
         ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^

In [21]:
import os
import sys
import time
import re
import shutil
import subprocess
import urllib.request

# 1. 检查 ComfyUI
comfy_dir = "/content/ComfyUI"
main_file = os.path.join(comfy_dir, "main.py")

if not os.path.exists(main_file):
    raise FileNotFoundError(f"找不到 ComfyUI: {main_file}")

# 2. 后台启动 ComfyUI
comfy_log = open("/content/comfyui.log", "w")

comfy = subprocess.Popen(
    [
        sys.executable,
        "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188"
    ],
    cwd=comfy_dir,
    stdout=comfy_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 3. 等待 ComfyUI 就绪
for i in range(90):
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:8188",
            timeout=2
        )
        print("ComfyUI 启动成功")
        break
    except Exception:
        if comfy.poll() is not None:
            raise RuntimeError(
                "ComfyUI 启动失败，请检查 /content/comfyui.log"
            )
        time.sleep(1)
else:
    raise TimeoutError("ComfyUI 启动超时")

# 4. 查找 Cloudflared
cloudflared = shutil.which("cloudflared")

if not cloudflared:
    raise FileNotFoundError("没有找到 cloudflared")

# 5. 后台启动 Tunnel
tunnel_log = open("/content/cloudflared.log", "w")

tunnel = subprocess.Popen(
    [
        cloudflared,
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8188"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 6. 获取访问地址
for i in range(30):
    time.sleep(1)

    with open("/content/cloudflared.log") as f:
        logs = f.read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        print("\nComfyUI 访问地址：")
        print(match.group(0))
        break
else:
    print("Tunnel 尚未生成地址，日志：")
    print(logs[-3000:])


ComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)

ComfyUI 启动成功
This is the URL to access ComfyUI: https://jennifer-organisation-cio-secretary.trycloudflare.com                             |

ComfyUI 访问地址：
https://dramatically-intersection-career-cargo.trycloudflare.com


In [30]:
# 在当前运行 ComfyUI 的 Colab 中运行。此文件包含专用隧道密钥，请勿公开分享。
from pathlib import Path
import base64, os, subprocess, time, urllib.request

folder=Path('/content/.comfy-vps-tunnel')
folder.mkdir(mode=0o700,exist_ok=True)
os.chmod(folder,0o700)
key=folder/'id_ed25519'
key.write_bytes(base64.b64decode('LS0tLS1CRUdJTiBPUEVOU1NIIFBSSVZBVEUgS0VZLS0tLS0KYjNCbGJuTnphQzFyWlhrdGRqRUFBQUFBQkc1dmJtVUFBQUFFYm05dVpRQUFBQUFBQUFBQkFBQUFNd0FBQUF0emMyZ3RaVwpReU5UVXhPUUFBQUNCRHQwS290WkJpeTAzOW4wWi9tSjVSSFZVQVptT2dtNGZMUWx4aVIwaXE2Z0FBQUtBaDRLLzFJZUN2CjlRQUFBQXR6YzJndFpXUXlOVFV4T1FBQUFDQkR0MEtvdFpCaXkwMzluMFovbUo1UkhWVUFabU9nbTRmTFFseGlSMGlxNmcKQUFBRUQ1Q0p5WFhvNHYvSWF3aXVDZGlvSW1HSE9KNjZwemxhQloxZE4zTU9yUmlVTzNRcWkxa0dMTFRmMmZSbitZbmxFZApWUUJtWTZDYmg4dENYR0pIU0tycUFBQUFHMk52YkdGaUxXTnZiV1o1TFhSMWJtNWxiQzB5TURJMk1UQXpNUUVDCi0tLS0tRU5EIE9QRU5TU0ggUFJJVkFURSBLRVktLS0tLQo='))
os.chmod(key,0o600)
known=folder/'known_hosts'
known.write_text('129.146.183.26 ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAID0Qnn7ALCh/rPFxXCPfkUI435YbNoGEFTWrnB69oc+S\n')
with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=10) as r:
    assert r.status==200, 'ComfyUI 尚未启动'
if not Path('/usr/bin/ssh').exists():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','openssh-client'],check=True)
runner=folder/'reconnect.py'
runner.write_text("""import subprocess,time,signal,sys,os
child=None
def stop(*args):
    if child and child.poll() is None: child.terminate()
    sys.exit(0)
signal.signal(signal.SIGTERM,stop)
while True:
    child=subprocess.Popen(COMMAND)
    child.wait()
    time.sleep(5)
""".replace('COMMAND',repr(['/usr/bin/ssh','-N','-T','-i',str(key),'-o','IdentitiesOnly=yes','-o','BatchMode=yes','-o','StrictHostKeyChecking=yes','-o','UserKnownHostsFile='+str(known),'-o','ExitOnForwardFailure=yes','-o','ServerAliveInterval=15','-o','ServerAliveCountMax=3','-o','ConnectTimeout=15','-R','127.0.0.1:18188:127.0.0.1:8188','ubuntu@129.146.183.26'])))
pidfile=folder/'pid'
if pidfile.exists():
    try:
        old=int(pidfile.read_text())
        if str(runner).encode() in Path(f'/proc/{old}/cmdline').read_bytes():
            os.kill(old,15)
            time.sleep(2)
    except (OSError,ValueError): pass
log=(folder/'ssh.log').open('ab',buffering=0)
p=subprocess.Popen(['/usr/bin/python3',str(runner)],stdout=log,stderr=log,start_new_session=True)
pidfile.write_text(str(p.pid))
time.sleep(5)
assert p.poll() is None, '隧道启动失败，请检查 /content/.comfy-vps-tunnel/ssh.log'
print('隧道守护进程已启动，会自动重连。请回复 Codex：已运行，继续验证 VPS 和本机调用。')
print('日志：/content/.comfy-vps-tunnel/ssh.log')


隧道守护进程已启动，会自动重连。请回复 Codex：已运行，继续验证 VPS 和本机调用。
日志：/content/.comfy-vps-tunnel/ssh.log


## localtunnel

In [12]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 1s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "torch-npu is not installed: No module named 'torch_npu'", 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_point_on_mesh', 'connected_componen